## Gold Layer Architecture

Using the Olist dataset, the recommended star schema is:

Gold Layer

Fact Tables
-----------
fact_orders
fact_sales
fact_payments

Dimension Tables
----------------
dim_customers
dim_products
dim_sellers
dim_date

### Step 1 — Create Gold Schema

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS main_workspace.gold
""")

### Step 2 — Load Silver Tables

In [0]:
customers = spark.table("main_workspace.silver.customers")
orders = spark.table("main_workspace.silver.orders")
order_items = spark.table("main_workspace.silver.order_items")
payments = spark.table("main_workspace.silver.payments")
products = spark.table("main_workspace.silver.products")
sellers = spark.table("main_workspace.silver.sellers")
category_translation = spark.table("main_workspace.silver.product_category_translation")

In [0]:
# Validate quickly:

display(customers)
display(orders)
display(order_items)

In [0]:
# Step 3 — Build Dimension Tables
# 3.1 Customer Dimension

dim_customers = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

In [0]:
dim_customers.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_customers")

In [0]:
# 3.2 Product Dimension

# We enrich products with translated category names.

dim_products = products.join(
    category_translation,
    "product_category_name",
    "left"
)

In [0]:
# %sql
# Select required columns:

dim_products = dim_products.select(
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

In [0]:
# Write:

dim_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_products")

In [0]:
# 3.3 Sellers Dimension
dim_sellers = sellers.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
)

In [0]:
# Write:

dim_sellers.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_sellers")

In [0]:
# Step 4 — Build Date Dimension

# Extract dates from order timestamps.

from pyspark.sql.functions import *

dim_date = orders.select(
    col("order_purchase_timestamp").alias("date")
).distinct()

In [0]:
# Add useful analytics columns:

dim_date = dim_date \
.withColumn("year", year("date")) \
.withColumn("month", month("date")) \
.withColumn("day", dayofmonth("date")) \
.withColumn("week", weekofyear("date"))

In [0]:
# Write:

dim_date.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_date")

In [0]:
# Step 5 — Build Fact Tables
# 5.1 Fact Orders
fact_orders = orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date"
)

In [0]:
# Write:

fact_orders.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.fact_orders")

In [0]:
# 5.2 Fact Sales

# Sales come from order_items.

fact_sales = order_items.select(
    "order_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value"
)

fact_orders.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.fact_orders")

In [0]:
# Write:

fact_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.fact_sales")

In [0]:
# 5.3 Fact Payments
fact_payments = payments.select(
    "order_id",
    "payment_type",
    "payment_installments",
    "payment_value"
)

In [0]:
# Write:

fact_payments.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.fact_payments")

## Step 1 — Load Current Gold Customer Dimension

In [0]:
from pyspark.sql.functions import *

dim_customers = spark.table("main_workspace.gold.dim_customers")

display(dim_customers)

In [0]:
dim_customers.printSchema()

In [0]:
# Step 3 — Create Surrogate Key


from pyspark.sql.functions import monotonically_increasing_id

dim_customers_scd2 = dim_customers \
.withColumn("customer_key", monotonically_increasing_id())

In [0]:
# Add SCD Columns
dim_customers_scd2 = dim_customers_scd2 \
.withColumn("effective_start_date", current_timestamp()) \
.withColumn("effective_end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True))

In [0]:
# Step 5 — Reorder Columns (Clean Design)
dim_customers_scd2 = dim_customers_scd2.select(
"customer_key",
"customer_id",
"customer_unique_id",
"customer_zip_code_prefix",
"customer_city",
"customer_state",
"effective_start_date",
"effective_end_date",
"is_current"
)

In [0]:
# Step 6 — Write Back to Gold Layer

# Overwrite the dimension table.

dim_customers_scd2.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_customers")

# Now your dimension table supports SCD Type-2.

In [0]:
# Step 1 — Drop the existing dimension table
spark.sql("""
DROP TABLE main_workspace.gold.dim_customers
""")

In [0]:
# Step 2 — Write the SCD2 version

# Now run your write again.

dim_customers_scd2.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_customers")

# This will create the table with the new SCD schema.

In [0]:
# Verify
display(
spark.table("main_workspace.gold.dim_customers")
)

## Checking SCD type 2 


In [0]:
from pyspark.sql.functions import *

dim_customers = spark.table("main_workspace.gold.dim_customers")

display(dim_customers)

In [0]:
# Step 2 — Select a Customer to Demonstrate

# We pick one customer from the table.

sample_customer = dim_customers.limit(1)

display(sample_customer)

In [0]:
# Extract the ID:

customer_id_value = sample_customer.select("customer_id").first()[0]
print(customer_id_value)

In [0]:
# Step 3 — Simulate a Change (Customer Moves City)

# Create a new version of the same customer.

changed_customer = sample_customer \
.withColumn("customer_city", lit("rio_de_janeiro")) \
.withColumn("customer_state", lit("RJ"))

In [0]:
# Display:

display(changed_customer)

In [0]:
# Step 4 — Expire the Old Record

# Old record must become inactive.

expired_records = dim_customers \
.filter(col("customer_id") == customer_id_value) \
.withColumn("effective_end_date", current_timestamp()) \
.withColumn("is_current", lit(False))

In [0]:
# Step 5 — Create the New Version
from pyspark.sql.functions import monotonically_increasing_id

new_version = changed_customer \
.withColumn("customer_key", monotonically_increasing_id()) \
.withColumn("effective_start_date", current_timestamp()) \
.withColumn("effective_end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True))

In [0]:
# Step 6 — Rebuild the Dimension Table

# Remove old active record and append both versions.

remaining_records = dim_customers.filter(col("customer_id") != customer_id_value)

final_dim_customers = remaining_records \
.unionByName(expired_records) \
.unionByName(new_version)

In [0]:
# Check result:

display(final_dim_customers.filter(col("customer_id") == customer_id_value))

In [0]:
from pyspark.sql.functions import *

# Load current table
dim_customers = spark.table("main_workspace.gold.dim_customers")

# Use same customer
customer_id_value = dim_customers.select("customer_id").first()[0]

# Current active record
active_record = dim_customers.filter(
    (col("customer_id") == customer_id_value) &
    (col("is_current") == True)
)

# Expire current record
expired_record = active_record \
.withColumn("effective_end_date", current_timestamp()) \
.withColumn("is_current", lit(False))

# Create another new version (third history record)
new_version = active_record \
.withColumn("customer_city", lit("salvador")) \
.withColumn("customer_state", lit("BA")) \
.withColumn("effective_start_date", current_timestamp()) \
.withColumn("effective_end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True)) \
.withColumn("customer_key", monotonically_increasing_id())

# Remove current record from table
remaining_records = dim_customers.filter(~(
    (col("customer_id") == customer_id_value) &
    (col("is_current") == True)
))

# Rebuild dimension
final_dim_customers = remaining_records \
.unionByName(expired_record) \
.unionByName(new_version)

# Save table
final_dim_customers.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.gold.dim_customers")

# Show history
display(
spark.table("main_workspace.gold.dim_customers")
.filter(col("customer_id") == customer_id_value)
.orderBy("effective_start_date")
)

Second Run
city_name = "curitiba"
state_name = "PR"

In [0]:
display(
spark.table("main_workspace.gold.dim_customers")
)

In [0]:
# %sql
# DESCRIBE HISTORY main_workspace.gold.dim_customers;

In [0]:
# %sql
# RESTORE TABLE main_workspace.gold.dim_customers TO VERSION AS OF 2;

In [0]:
# %sql
# SELECT *
# FROM main_workspace.gold.dim_customers


In [0]:
display(
spark.table("main_workspace.gold.dim_customers").count()
)

In [0]:
display(
spark.table("main_workspace.gold.dim_customers")

)

In [0]:
display(
spark.table("main_workspace.gold.dim_customers")
.limit(5)
)

In [0]:
# Step 1 — Load table
from pyspark.sql.functions import *

dim_customers = spark.table("main_workspace.gold.dim_customers")

customer_id_value = "000598caf2ef4117407665ac33275130"

In [0]:
# Step 2 — Identify current row
active_row = dim_customers.filter(
    (col("customer_id") == customer_id_value) &
    (col("is_current") == True)
)

display(active_row)

In [0]:
# Step 3 — Expire that row (same key)
expired_row = active_row \
.withColumn("effective_end_date", current_timestamp()) \
.withColumn("is_current", lit(False))

In [0]:
# Step 4 — Create new version (new key only here)
max_key = dim_customers.agg(max("customer_key")).collect()[0][0]

new_row = active_row \
.withColumn("customer_key", lit(max_key + 1)) \
.withColumn("customer_city", lit("rio_de_janeiro")) \
.withColumn("customer_state", lit("RJ")) \
.withColumn("effective_start_date", current_timestamp()) \
.withColumn("effective_end_date", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True))

In [0]:
# Step 5 — Update dataset (no deletion of key 1)
updated_table = dim_customers \
.withColumn(
    "is_current",
    when(
        (col("customer_id") == customer_id_value) & (col("is_current") == True),
        False
    ).otherwise(col("is_current"))
).withColumn(
    "effective_end_date",
    when(
        (col("customer_id") == customer_id_value) & (col("is_current") == False),
        current_timestamp()
    ).otherwise(col("effective_end_date"))
)

updated_table = updated_table.unionByName(new_row)

In [0]:
# Verify before writing
display(
updated_table
.filter(col("customer_id") == customer_id_value)
.orderBy("customer_key")
)

In [0]:
print("Old table count:", dim_customers.count())
print("Updated table count:", updated_table.count())

In [0]:
display(
updated_table
.filter(col("customer_id")==customer_id_value)
.select("customer_id","customer_city","is_current")
)

In [0]:
display(
updated_table
.filter(col("customer_id") == "000598caf2ef4117407665ac33275130")
)

Databricks filtered table. Run in Databricks to view.

In [0]:
# Verify before writing
updated_table.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("main_workspace.gold.dim_customers")